# 核心代码速查宝典 (Core Code Cheat Sheet)

在我们的项目中，你接触了目前大模型领域最前沿的四大底层框架：`Transformers` (大模型底座)、`LangChain` (数据与检索框架)、`PEFT & TRL` (高效微调框架) 以及 `Gradio` (交互界面)。

为了帮你巩固记忆，我把整个项目中最具“含金量”和“复用价值”的核心代码段全部抽取了出来，形成这份备忘录。以后你在公司做其他 AI 项目时，直接来这里“抄作业”即可。

---

## 1. 大模型的加载与对话生成 (Transformers 核心)
这是驱动所有文本大模型 (如 LLaMA, Qwen, ChatGLM) 的万能公式。




In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 1. 显存节约技巧：一定要带上 torch_dtype=torch.float16 (半精度) 和 device_map="auto" (自动分配显存)
tokenizer = AutoTokenizer.from_pretrained("模型路径")
model = AutoModelForCausalLM.from_pretrained(
    "模型路径", 
    torch_dtype=torch.float16, 
    device_map="auto"
)

# 2. 组装对话格式 (这是新版模型最重要的机制)
messages = [
    {"role": "system", "content": "你是一个助手"},
    {"role": "user", "content": "你好"}
]
# apply_chat_template 会自动根据不同模型的要求，把字典拼接成特定的 Prompt 字符串
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 3. 转化为模型认识的张量 (Tensor) 并推给 GPU
inputs = tokenizer([text], return_tensors="pt").to("cuda")

# 4. 生成回复并解码为人类文字
generated_ids = model.generate(**inputs, max_new_tokens=256)
# 这一步是为了截断掉前面的提问，只保留模型新生成的话
gen_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
final_text = tokenizer.batch_decode(gen_trimmed, skip_special_tokens=True)[0]


---

## 2. RAG 本地知识库的构建与检索 (LangChain 核心)
告别手写向量计算，用 LangChain 的封装备件快速搭建本地检索大脑。




In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 加载本地向量大模型 (BGE 等)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")

# 2. 绑定或新建本地数据库 (persist_directory 会把数据存在本地文件夹)
vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

# 3. 存入数据 (Data Ingestion)
vector_db.add_texts(
    texts=["这是第一条知识", "这是第二条知识"],
    metadatas=[{"source": "书籍1"}, {"source": "书籍2"}]
)

# 4. 检索数据 (Retrieval)
# k=2 表示只拿最相似的前 2 条内容
docs = vector_db.similarity_search("搜索词", k=2)
for doc in docs:
    print(doc.page_content) # 文本内容
    print(doc.metadata)     # 溯源信息


---

## 3. LoRA 轻量级微调大模型 (PEFT 核心)
不破坏原有大模型，只在外面套一层可以训练的“外挂”。




In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# 1. 配置 LoRA 插件 (决定微调哪些层，r 和 alpha 是核心强度参数)
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # 通常盯着大模型的 Attention 层打
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. 将外挂挂载到原模型上
model = get_peft_model(model, lora_config)

# 3. 新版本 TRL 规定的标准训练参数格式
training_args = SFTConfig(
    output_dir="./lora_output",
    per_device_train_batch_size=1, # 显存小就开1
    learning_rate=2e-4,            # LoRA 的学习率通常比全量微调大一点
    num_train_epochs=3
)

# 4. 启动训练
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset, # 读取的 JSONL 数据
    args=training_args
)
trainer.train()

# 5. 保存外挂权重
trainer.save_model("./my_lora_adapter")


---

## 4. 多模态视觉大模型看图 (Qwen-VL 核心)
如何把图片塞进提示词中。




In [ ]:
from qwen_vl_utils import process_vision_info

# 与纯文本不同，多模态模型的 message 里的 content 变成了一个“列表”
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "test_chart.png"}, # 指定图片路径
            {"type": "text", "text": "请描述图片内容"}
        ],
    }
]

# process_vision_info 是专用工具，用于把字典里的图片路径提取并转化为像素张量
image_inputs, video_inputs = process_vision_info(messages)

# 丢进处理器
inputs = processor(
    text=[text], 
    images=image_inputs, 
    padding=True, 
    return_tensors="pt"
).to("cuda")


---

## 5. 多轮对话 RAG 的上下文拼接技巧 (工程化思维)
RAG 最大的坑就是用户问“它的算力呢？”，数据库因为没有主语而搜不到。




In [ ]:
# 暴力但极其有效的拼接法
search_query = message # 用户当前问题
if len(history) > 0:
    # 获取上一轮的历史记录
    last_item = history[-1] 
    
    # 根据框架格式提取上一轮文本（应对 Gradio 的各类格式）
    last_msg = last_item["content"] if isinstance(last_item, dict) else last_item[0]
    
    # 将上一轮提问与这一轮提问无脑拼接，大大增加向量检索的命中率！
    search_query = str(last_msg) + " " + str(message)

# 再拿去搜索
docs = vector_db.similarity_search(search_query, k=2)
